## Imports and set constants

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit

TRAIN_DATA_PATH = "data/train_team_track.parquet"
TEST_DATA_PATH = "data/test_team_track.parquet"

RANDOM_STATE = 42

## Read and data processing

In [ ]:
# Загрузка
train = pd.read_parquet(TRAIN_DATA_PATH)

# Обработка времени, извлекаем все признаки, которые можем из даты
train["timestamp"] = pd.to_datetime(train["timestamp"])

train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

## Достаём доп фичи из даты
## day_of_week, day_of_month для древовыдных моделей не страшны
## Месяц кажется, что не сильно важнен:
## В данных всего 2 месяца, скорее доп признак будет просто шум, ничего полезного
train["day_of_week"] = train["timestamp"].dt.dayofweek
train["is_weekend"] = (train["timestamp"].dt.dayofweek >= 5).astype(int)
train["time"] = train["timestamp"].dt.time
train["day_of_month"] = train["timestamp"].dt.day

## Циклическое кодирование времени в течение дня
time_parsed = pd.to_datetime(train["time"], format="%H:%M:%S")
seconds_in_day = (
    time_parsed.dt.hour * 3600
    + time_parsed.dt.minute * 60
    + time_parsed.dt.second
)

day_seconds = 24 * 60 * 60
train["time_sin"] = np.sin(2 * np.pi * seconds_in_day / day_seconds)
train["time_cos"] = np.cos(2 * np.pi * seconds_in_day / day_seconds)

# Обработка route_id: Достанем доп фичи: lags по таргету + rolling так же + оставим как есть route_id
# Пока только для train
lag_list = [1, 2, 4, 6, 12, 24, 48, 96, 144]
rolling_windows = [3, 6, 12, 24, 48, 96]

## Делаем lags
grouped = train.groupby("route_id")["target_2h"]
for lag in lag_list:
    train[f"target_2h_lag_{lag}"] = grouped.shift(lag)

## Теперь rolling
shifted = grouped.shift(1)

## В первом варианте достаточно много статистик. 
## Посмотрим, что сможет выжить catboost
for window in rolling_windows:
    train[f"target_2h_roll_mean_{window}"] = (
        shifted.rolling(window=window).mean()
        .reset_index(level=0, drop=True)
    )
    train[f"target_2h_roll_std_{window}"] = (
        shifted.rolling(window=window).std()
        .reset_index(level=0, drop=True)
    )
    # train[f"target_2h_roll_min_{window}"] = (
    #     shifted.rolling(window=window).min()
    #     .reset_index(level=0, drop=True)
    # )
    # train[f"target_2h_roll_max_{window}"] = (
    #     shifted.rolling(window=window).max()
    #     .reset_index(level=0, drop=True)
    # )
    # train[f"target_2h_roll_median_{window}"] = (
    #     shifted.rolling(window=window).median()
    #     .reset_index(level=0, drop=True)
    # )

# Пока выкинем статусы, не совсем понимаю, как потом с ними работать.
train = train.drop(
    [
        "status_1", 
        "status_2", 
        "status_3", 
        "status_4", 
        "status_5", 
        "status_6", 
        "status_7", 
        "status_8",
        "office_from_id",
        "time"
    ], 
    axis=1
)

# Уменьшаем количество занимаемой памяти
# По int точно нет потерь, значения маленькие
# По флоат, ну думаю будет достаточно
for col in train.select_dtypes(include=['int64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='integer')

for col in train.select_dtypes(include=['float64']).columns:
    train[col] = pd.to_numeric(train[col], downcast='float')

# Улаляю лишнее
del(time_parsed)
del(grouped)
del(shifted)

train = train.sort_values(["timestamp"]).reset_index(drop=True)
train = train.drop(["timestamp"], axis=1)

In [ ]:
train.info()

## Correlation

In [ ]:
corr = train.corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

## Hackathon metrics

In [ ]:
# Метрика хакатона
def wape_plus_rbias(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    denom = y_true.sum()
    if denom == 0:
        raise ValueError("y_true.sum() == 0, метрика не определена.")

    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = np.abs(y_pred.sum() / denom - 1.0)
    return wape + rbias

## Cross validation (optional)

In [ ]:
# Кросс валидация для проверки качества модели
# Веса от неё никак не сохраняются, но можно посмотреть,
# а норм ли идея.

# Для использования, нужно раскоментить следующий код в ячейке

# X = train.drop(["target_2h"], axis=1).reset_index(drop=True)
# y = train["target_2h"].reset_index(drop=True)

# cat_feature = ["route_id", "day_of_week", "is_weekend", "day_of_month"]

# tscv = TimeSeriesSplit(
#     n_splits=5,
#     test_size=1000 * 2 * 48,
#     max_train_size=1000 * 14 * 48,
#     gap=0,
# )

# scores = []

# for fold, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):
#     X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
#     y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

#     model = CatBoostRegressor(
#         loss_function="MAE",
#         eval_metric="MAE",
#         iterations=2000,
#         learning_rate=0.02,
#         depth=6,
#         l2_leaf_reg=5,
#         random_seed=RANDOM_STATE,
#         verbose=200,
#     )

#     model.fit(
#         X_train,
#         y_train,
#         cat_features=cat_feature,
#         eval_set=(X_valid, y_valid),
#         use_best_model=True,
#         early_stopping_rounds=200,
#     )

#     pred = np.clip(model.predict(X_valid), 0, None)
#     score = wape_plus_rbias(y_valid.values, pred)
#     scores.append(score)

#     print(f"Fold {fold}: {score:.6f}")

# print("Mean CV:", np.mean(scores))

## Train model

In [ ]:
X = train.drop(["target_2h"], axis=1).reset_index(drop=True)
y = train["target_2h"].reset_index(drop=True)

cat_feature = ["route_id", "day_of_week", "is_weekend", "day_of_month"]


model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=2000,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=200,
)

model.fit(
    X,
    y,
    cat_features=cat_feature,
    use_best_model=True,
    early_stopping_rounds=200,
)

model.save_model("catboost_model.cbm")


## Marking up test data

In [ ]:
model = CatBoostRegressor()
model.load_model("catboost_model.cbm")

# История таргета из train для построения lag/rolling на test
history_df = pd.read_parquet(TRAIN_DATA_PATH, columns=["route_id", "timestamp", "target_2h"])
history_df["timestamp"] = pd.to_datetime(history_df["timestamp"])
history_df = history_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# Готовим test
test_pred = pd.read_parquet(TEST_DATA_PATH).copy()
test_pred["timestamp"] = pd.to_datetime(test_pred["timestamp"])
test_pred = test_pred.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# Храним историю по каждому маршруту, затем будем дописывать предсказания
route_history = {
    int(route_id): grp["target_2h"].to_list()
    for route_id, grp in history_df.groupby("route_id", sort=False)
}

feature_cols = X.columns.tolist()
preds = np.zeros(len(test_pred), dtype=np.float32)

for i, row in enumerate(test_pred.itertuples(index=False)):
    rid = int(row.route_id)
    ts = row.timestamp

    h = route_history[rid]

    row_features = {
        "route_id": rid,
        "day_of_week": int(ts.dayofweek),
        "is_weekend": int(ts.dayofweek >= 5),
        "day_of_month": int(ts.day),
    }

    sec = ts.hour * 3600 + ts.minute * 60 + ts.second
    row_features["time_sin"] = np.sin(2 * np.pi * sec / 86400)
    row_features["time_cos"] = np.cos(2 * np.pi * sec / 86400)

    for lag in lag_list:
        row_features[f"target_2h_lag_{lag}"] = h[-lag] if len(h) >= lag else np.nan

    for w in rolling_windows:
        if len(h) >= w:
            window_vals = np.asarray(h[-w:], dtype=np.float32)
            row_features[f"target_2h_roll_mean_{w}"] = float(window_vals.mean())
            row_features[f"target_2h_roll_std_{w}"] = float(window_vals.std(ddof=1)) if w > 1 else 0.0
        else:
            row_features[f"target_2h_roll_mean_{w}"] = np.nan
            row_features[f"target_2h_roll_std_{w}"] = np.nan

    row_df = pd.DataFrame([row_features], columns=feature_cols)
    pred = float(model.predict(row_df)[0])
    pred = max(pred, 0.0)

    preds[i] = pred
    h.append(pred)

# Добавляем предикт в test и сохраняем
test_marked = test_pred.copy()
test_marked["y_pred"] = preds

submission = test_marked[["id", "y_pred"]].copy()
submission.to_csv("submission.csv", index=False)

print(test_marked.head())
print("Saved: submission.csv")